# CS-781 Capstone — Phase 1: Decoder-Only Transformer

This notebook trains and submits ONE architecture end-to-end on Kaggle:
1. Get the legal text data
2. Turn text into numbers (a BPE subword tokenizer, trained on this corpus)
3. Build the Decoder-Only Transformer
4. Train it (checkpoints saved to `/kaggle/working`)
5. Generate the 128-word continuations for the test set
6. Package `submission.zip` to upload to Codabench

**Workflow:** run this notebook with `QUICK_TEST = True` first (Settings
cell) to confirm everything works on a small slice of data in a few
minutes. Then set `QUICK_TEST = False` and run it again for real. This
is one of 4 separate notebooks (decoder-only Transformer, encoder-decoder
Transformer, LSTM, CNN) — the brief requires comparing 4 substantially
different architectures, so run all 4 the same way and compare their
local evaluation scores (Step 15) before deciding what to submit.

On top of that, this notebook also exposes `NORM_TYPE`, `NORM_POSITION`, `ATTENTION_TYPE` and `OPTIMIZER_TYPE` in the Settings cell, so you can systematically try
different normalization / attention / optimizer choices (per the lecture
slides on comparing attentions and normalizations) and compare them the
same way you compare architectures.

**Before running, in the panel on the right:**
- **Accelerator** → pick a GPU (e.g. "GPU T4 x2").
- **Internet** → turn ON (needed to download the dataset from Hugging Face
  and to `pip install` a couple of packages).

## 1. Settings

In [1]:
# ----------------------------------------------------------------
# QUICK_TEST lets you sanity-check the whole notebook in a few
# minutes before committing GPU time to a real run.
#
#   1. First run: leave QUICK_TEST = True. Run every cell top to
#      bottom. This trains on a small slice of data for 1 epoch,
#      just to prove nothing crashes and a submission.zip comes out
#      the other end.
#   2. Once that works: set QUICK_TEST = False and re-run every cell.
#      This is your real run, on much more data, for more epochs.
# ----------------------------------------------------------------
QUICK_TEST = False

if QUICK_TEST:
    MAX_WORDS_FOR_TRAINING = 200_000   # a small slice, just to test the pipeline
    NUM_EPOCHS = 1
else:
    MAX_WORDS_FOR_TRAINING = 60000000   # increase this further if your GPU/time budget allows
    NUM_EPOCHS = 2

VOCAB_SIZE = 40000        # BPE subword vocabulary size -- larger than a word-level vocab needs to be, since subwords cover rare words without <unk>
CONTEXT_LEN = 640        # 512 (the prompt) + 128 (the continuation)
BATCH_SIZE = 8           # how many examples we train on at once
LEARNING_RATE = 3e-4
TEMPERATURE = 0.0        # 0 = always pick the most likely next word (greedy)
                         # try 0.7 or 1.0 later for more varied/creative text

# ----------------------------------------------------------------
# Research knobs -- this is the whole point of a research project:
# change ONE of these, re-run, and compare. See lecture slides on
# "Comparing Attentions" and normalization for background.
# ----------------------------------------------------------------
NORM_TYPE = "layernorm"  # "layernorm" | "rmsnorm" | "deepnorm"
# layernorm = the standard normalization used in most Transformers
# rmsnorm   = a cheaper norm that skips re-centering the mean (used in
#             LLaMA and others)
# deepnorm  = a post-norm recipe designed to make very deep stacks stable
#             (scales the residual connection by a fixed alpha and uses a
#             special weight init). deepnorm always places the norm AFTER
#             the sub-layer (it ignores NORM_POSITION below).

NORM_POSITION = "pre"    # "pre" | "post"
# pre  = normalize BEFORE each sub-layer (norm(x) -> sublayer -> add residual).
#        This is what most modern Transformers (GPT-2 and later) use --
#        it trains more stably, especially for deep models.
# post = normalize AFTER adding the residual (sublayer -> add -> norm(...)).
#        This is what the original 2017 Transformer paper used.

ATTENTION_TYPE = "mha"  # "mha" | "gqa" | "mqa"
# mha = every query head gets its own key/value head (the standard design)
# gqa = query heads are split into groups that SHARE one key/value head
#       (fewer key/value heads -> less memory, from the "grouped-query
#       attention" lecture slide)
# mqa = the extreme version of gqa: ALL query heads share a single
#       key/value head ("multi-query attention")

OPTIMIZER_TYPE = "adamw"  # "adamw" | "muon"
# adamw = the standard optimizer almost everyone uses.
# muon  = a newer optimizer that "orthogonalizes" the update for every 2D+
#         weight matrix (biases/embeddings/norm weights still use adamw
#         underneath -- that's the standard muon recipe).

# Checkpoint filename includes QUICK_TEST, NORM_TYPE/POSITION, ATTENTION_TYPE
# and OPTIMIZER_TYPE so different experiment settings never overwrite each
# other's checkpoints (this also means switching ATTENTION_TYPE between runs
# always starts a fresh checkpoint instead of colliding with a differently-
# shaped one from another attention type).
mode_tag = "quicktest" if QUICK_TEST else "full"
CHECKPOINT_PATH = (
    f"/kaggle/working/phase1_decoder_only_{mode_tag}_{NORM_TYPE}_{NORM_POSITION}"
    f"_{ATTENTION_TYPE}"
    f"_{OPTIMIZER_TYPE}_checkpoint.pt"
)

## 2. Make sure the working folder exists (and a note on Kaggle persistence)

Checkpoints go to `/kaggle/working`. That folder survives you re-running
cells, and even a "Restart session", within the same notebook edit session.

If you come back on a **completely new day / new session** and want to
resume: click **Save Version** before you stop working. Next time, use
**Add Input → Notebook Output Files → (this notebook's last saved version)**
to pull the checkpoint back in, then copy it into `/kaggle/working` before
re-running the training cell.

In [2]:
import os
os.makedirs("/kaggle/working", exist_ok=True)
print("Working folder ready:", os.path.exists("/kaggle/working"))

Working folder ready: True


## 3. One-time setup, then install and import what we need

**The dataset is GATED** -- before this notebook can download anything, do
this once per Kaggle **account** (each of your two accounts needs its own):

1. While logged into Hugging Face, open
   https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone and
   accept the dataset's access conditions (there's a form to agree to on
   that page -- without this, downloads below get an auth error).
2. Create a **read** access token at https://huggingface.co/settings/tokens.
3. In this Kaggle notebook: **Add-ons → Secrets → Add a new secret**, name
   it exactly `HF`, and paste the token as its value.

Do this before running the cell below -- it logs in using that secret.

In [3]:
import json
import os
import zipfile
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

Using device: cuda


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 3b. Sync checkpoints with a Hugging Face model repo

Instead of manually downloading/uploading `.pt` files between Kaggle
sessions or accounts, checkpoints are pushed to a Hugging Face **model**
repo after every epoch, and pulled back down automatically if this session
doesn't already have a local one. The filename already encodes the full
config (architecture, QUICK_TEST mode, norm/attention/optimizer settings --
see the Settings cell), so the repo doubles as a self-documenting archive
of every run you've tried -- handy when writing up the report.

**One-time setup (separate from the read-only `HF` secret used for the
gated dataset):**
1. Create a Hugging Face **model** repo (e.g. at
   huggingface.co/new -> "Model"), private or public, your choice.
2. Create a **write**-scoped access token at
   https://huggingface.co/settings/tokens.
3. In this Kaggle notebook: **Add-ons -> Secrets -> Add a new secret**,
   name it exactly `HF_WRITE`, and paste that token as its value. (Keep
   the existing `HF` secret as-is -- that one stays read-only, just for
   downloading the dataset.)

In [4]:
from huggingface_hub import HfApi, create_repo
import shutil

HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
hf_api = HfApi(token=HF_WRITE_TOKEN)
create_repo(HF_CHECKPOINT_REPO, token=HF_WRITE_TOKEN, repo_type="model", exist_ok=True)


def push_checkpoint_to_hub(local_path):
    """Upload the checkpoint at local_path to the HF repo, under its own
    filename (which already encodes the full config -- see Settings)."""
    hf_api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=os.path.basename(local_path),
        repo_id=HF_CHECKPOINT_REPO,
        repo_type="model",
        token=HF_WRITE_TOKEN,
    )
    print(f"Pushed checkpoint to https://huggingface.co/{HF_CHECKPOINT_REPO}"
          f"/blob/main/{os.path.basename(local_path)}")


def try_pull_checkpoint_from_hub(local_path):
    """If no LOCAL checkpoint exists yet (e.g. a fresh Kaggle session or a
    different account), try to fetch a matching one from the HF repo
    before falling back to starting from scratch."""
    if os.path.exists(local_path):
        return
    try:
        downloaded_path = hf_hub_download(
            repo_id=HF_CHECKPOINT_REPO, filename=os.path.basename(local_path),
            repo_type="model", token=HF_WRITE_TOKEN,
        )
        shutil.copy(downloaded_path, local_path)
        print(f"Pulled matching checkpoint from the Hub: {os.path.basename(local_path)}")
    except Exception as e:
        print(f"No matching checkpoint on the Hub yet ({type(e).__name__}) -- starting fresh.")


print(f"Checkpoint sync configured -> https://huggingface.co/{HF_CHECKPOINT_REPO}")

Checkpoint sync configured -> https://huggingface.co/kavincg23/cs781-checkpoints


## 4. Download the data files from Hugging Face

This dataset has no `datasets.load_dataset(...)` config -- it's raw JSON
files, one per source, downloaded directly (the same way the official
starter notebook does it). Make sure **Internet** is turned ON in the
settings panel on the right.

**Important:** the 4 train files together hold the *entire* >4.5B-token
corpus. `json.load()`-ing even one of them fully into memory is enough to
exceed Kaggle's RAM and crash the notebook ("tried to allocate more memory
than is available") -- and that happens regardless of `MAX_WORDS_FOR_TRAINING`,
since that setting only limits how many words we *use*, not how much we'd
have loaded first. So instead of `json.load`, we use `ijson` to stream
through each file's JSON array one document at a time, only ever holding a
handful of documents in memory. The print statement below just peeks at
the first document, to let you SEE the real shape of the data.

In [5]:
!pip install -q ijson
import ijson

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

first_train_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TRAIN_FILES[0],
                                    repo_type="dataset", token=HF_TOKEN)
with open(first_train_path, "r", encoding="utf-8") as f:
    preview_record = next(ijson.items(f, "item"))
print("Example document (from the first train file):")
print(preview_record)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 3.9 MB/s eta 0:00:00


Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

Example document (from the first train file):
{'title': 'Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996', 'category': 'Greater Bengaluru City Corporation - Act', 'year': 1972, 'url': 'https://indiankanoon.org/doc/179573747/', 'text': 'Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors including ground floor. The Corporation of the City of Bangalore granted permission for construction of buildings beyond five floors in certain cases. In respect of some of the buildings the permission granted by the Corporation contrary to the Zonal Regulations then in force was questioned in the High Court and the Division Bench of the High Court held that the permission granted in violation of the Zonal Regulation was illegal.

### Check the field name above

Look at the example printed above -- it should be a dict like
`{'text': '...'}` (confirmed from the official starter notebook). Set
`TEXT_FIELD` below to whatever key actually holds the legal text --
change it if the printed example above used a different name.

In [6]:
TEXT_FIELD = "text"  # <-- change this if needed, based on what you saw above

### Download and load the test prompts (the ones we must generate continuations for)

`test.json` is a single JSON object keyed by ID (`{"1": {"text": "..."}, ...}`),
not a list -- that's already the exact shape Codabench wants predictions in.

In [7]:
test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE,
                                  repo_type="dataset", token=HF_TOKEN)
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_prompts = {str(doc_id): record[TEXT_FIELD] for doc_id, record in test_data.items()}

print(f"Loaded {len(test_prompts)} test prompts")
first_id = list(test_prompts.keys())[0]
print(f"Example prompt (id={first_id}), word count = {len(test_prompts[first_id].split())}")

test.json: 0.00B [00:00, ?B/s]

Loaded 1000 test prompts
Example prompt (id=1), word count = 512


### Stream through the training files, collecting words until we have enough

This walks through `TRAIN_FILES` one document at a time (still via `ijson`,
never loading a full file into memory) and stops as soon as it has collected
`MAX_WORDS_FOR_TRAINING` words -- so a `QUICK_TEST` run only ever reads a
small slice of the corpus off disk, not all >4.5B tokens of it.

In [8]:
print("Streaming through the training files, collecting words until we have enough...")

# Words are written straight to a FILE as we stream through the sources,
# instead of being collected into one giant Python list/string first. The
# BPE tokenizer below trains directly from this file (its Rust backend
# streams it from disk) rather than us handing it one huge in-memory
# string -- this is what lets tokenizer training handle a large word count
# without running out of RAM (building the string in Python first was the
# actual bottleneck, not the tokenizer trainer itself).
CORPUS_FILE = "/kaggle/working/bpe_training_corpus.txt"

if QUICK_TEST:
    # fast pipeline check -- just fill up to MAX_WORDS_FOR_TRAINING from
    # whichever source(s) come first, same as before, just writing to the
    # corpus file instead of a Python list.
    word_count = 0
    with open(CORPUS_FILE, "w", encoding="utf-8") as out_f:
        for filename in TRAIN_FILES:
            if word_count >= MAX_WORDS_FOR_TRAINING:
                break
            local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename,
                                          repo_type="dataset", token=HF_TOKEN)
            with open(local_path, "r", encoding="utf-8") as f:
                for record in ijson.items(f, "item"):
                    doc_words = record[TEXT_FIELD].split()
                    remaining = MAX_WORDS_FOR_TRAINING - word_count
                    if remaining <= 0:
                        break
                    if len(doc_words) > remaining:
                        doc_words = doc_words[:remaining]
                    out_f.write(" ".join(doc_words) + " ")
                    word_count += len(doc_words)
                    if word_count >= MAX_WORDS_FOR_TRAINING:
                        break
            print(f"  ...after {filename}: {word_count} words collected so far")
else:
    # Full run: pull a FIXED QUOTA from EACH of the 4 sources, instead of
    # filling up sequentially (which would silently train on only the
    # first source(s)). This is the "data mixing across sources"
    # experiment the brief expects -- change the quotas below to try a
    # different mix, and document what you tried.
    SOURCE_WORD_QUOTAS = {
    "Phase_1_Pretraining/Train/train_acts.json": 15000000,
    "Phase_1_Pretraining/Train/train_high_courts.json": 15000000,
    "Phase_1_Pretraining/Train/train_supreme_court.json": 15000000,
    "Phase_1_Pretraining/Train/train_tribunals.json": 15000000,
    }
    word_count = 0
    with open(CORPUS_FILE, "w", encoding="utf-8") as out_f:
        for filename in TRAIN_FILES:
            quota = SOURCE_WORD_QUOTAS.get(filename, 0)
            if quota <= 0:
                continue
            source_word_count = 0
            local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename,
                                          repo_type="dataset", token=HF_TOKEN)
            with open(local_path, "r", encoding="utf-8") as f:
                for record in ijson.items(f, "item"):
                    doc_words = record[TEXT_FIELD].split()
                    remaining = quota - source_word_count
                    if remaining <= 0:
                        break
                    if len(doc_words) > remaining:
                        doc_words = doc_words[:remaining]
                    out_f.write(" ".join(doc_words) + " ")
                    source_word_count += len(doc_words)
                    if source_word_count >= quota:
                        break
            word_count += source_word_count
            print(f"  ...collected {source_word_count} words from {filename} (target {quota})")
    MAX_WORDS_FOR_TRAINING = word_count  # actual total = sum of quotas above

print(f"\nUsing {word_count} words for training (written to {CORPUS_FILE})")

Streaming through the training files, collecting words until we have enough...
  ...collected 15000000 words from Phase_1_Pretraining/Train/train_acts.json (target 15000000)


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

  ...collected 15000000 words from Phase_1_Pretraining/Train/train_high_courts.json (target 15000000)


Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

  ...collected 15000000 words from Phase_1_Pretraining/Train/train_supreme_court.json (target 15000000)


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

  ...collected 15000000 words from Phase_1_Pretraining/Train/train_tribunals.json (target 15000000)

Using 60000000 words for training (written to /kaggle/working/bpe_training_corpus.txt)


## 5. Train a BPE (subword) tokenizer on the legal text

A word-level tokenizer with a small vocabulary throws away most rare words
as `<unk>` -- bad for legal text, which is full of case names, section
numbers, and specialized terms that don't repeat often enough to make a
small "most common words" list. **BPE (Byte-Pair Encoding)** instead
learns to break rare/long words into smaller reusable subword pieces, so
nothing needs to become `<unk>` at all -- a word that never appeared in
training can still be spelled out from pieces the tokenizer DID learn.

We train our OWN BPE tokenizer here (via Hugging Face's `tokenizers`
library) directly on this notebook's legal corpus, rather than reusing a
generic pretrained BPE vocabulary (like GPT-2's) -- so the learned subword
pieces reflect actual legal vocabulary patterns instead of generic web text.

In [9]:
!pip install -q tokenizers

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
import array

print("Training the BPE tokenizer...")
bpe_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_tokenizer.decoder = decoders.ByteLevel()

bpe_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=["<pad>", "<unk>"])

# Train directly from the corpus FILE on disk instead of handing the
# tokenizer one giant in-memory Python string. The tokenizers library's
# Rust backend streams the file itself, so the full 60M-word corpus never
# has to exist as one Python object at the same time the trainer is
# building its vocabulary -- this is what avoids the OOM kernel crash.
bpe_tokenizer.train([CORPUS_FILE], trainer=bpe_trainer)


def encode(text):
    """Turn a string into a list of BPE subword-token IDs."""
    return bpe_tokenizer.encode(text).ids


def decode(ids):
    """Turn a list of BPE subword-token IDs back into a string."""
    return bpe_tokenizer.decode(ids)


print("Vocabulary size:", bpe_tokenizer.get_vocab_size())

# Turn ALL of our collected training text into token IDs -- but stream this
# step too, the same way we streamed the corpus onto disk in the first
# place. A single encode() call over the full 60M-word string would build
# one giant tokenizers Encoding object holding several parallel arrays
# sized to the token count (token strings, character offsets, etc. -- not
# just the .ids we actually want), which is itself big enough to crash the
# kernel even though tokenizer TRAINING already succeeded. So instead we
# read the corpus file back in bounded-size pieces, encode each piece on
# its own (keeping only its .ids, discarding the rest of that piece's
# Encoding object before moving on), and store the running token stream in
# an array.array of packed C ints -- not a plain Python list, which would
# still cost one full Python object per token (tens of millions of them).
ENCODE_CHUNK_CHARS = 5_000_000
all_word_ids = array.array("i")
leftover = ""
with open(CORPUS_FILE, "r", encoding="utf-8") as f:
    while True:
        block = f.read(ENCODE_CHUNK_CHARS)
        if not block:
            break
        piece = leftover + block
        # don't encode a word that got cut in half at the chunk boundary --
        # hold the trailing partial word back for the next chunk instead.
        # Keep the SPACE with the leftover (not with this piece): the
        # byte-level BPE pre-tokenizer attaches a leading space to the word
        # that follows it, so a word encoded without its leading space is
        # tokenized differently than the same word mid-sentence -- keeping
        # the space attached to leftover is what makes chunked encoding
        # produce token-for-token identical output to encoding the whole
        # corpus in one call (verified against bpe_tokenizer.encode()).
        cut = piece.rfind(" ")
        if cut == -1:
            leftover = piece
            continue
        leftover = piece[cut:]
        piece = piece[:cut]
        all_word_ids.extend(bpe_tokenizer.encode(piece).ids)
if leftover.strip():
    all_word_ids.extend(bpe_tokenizer.encode(leftover).ids)

# (named all_word_ids to match the other notebooks -- with BPE it now holds
# subword-TOKEN ids rather than one id per word, chunking below doesn't care)
print(f"{word_count} words became {len(all_word_ids)} BPE tokens "
      f"({len(all_word_ids) / word_count:.2f} tokens per word)")

Training the BPE tokenizer...



Vocabulary size: 40000
60000000 words became 73744881 BPE tokens (1.23 tokens per word)


## 6. Chop the training text into fixed-size chunks

We cut the long stream of word-IDs into pieces of length `CONTEXT_LEN`
(640 words). Each piece is one training example. A model learns to predict
the next word at every position in the piece.

*Simplification:* this glues all documents together into one long stream
before cutting, so a few chunks will mix the end of one case with the start
of another. That's fine for a first version — separating documents cleanly
is one of the "task-specific preprocessing" experiments you could try later.

In [10]:
blocks = []
for start in range(0, len(all_word_ids) - CONTEXT_LEN, CONTEXT_LEN):
    # list(...) here works whether all_word_ids is a plain Python list
    # (word-level tokenizer) or an array.array of packed ints (BPE
    # tokenizer) -- either way each block becomes an ordinary list of ints
    # before torch.tensor(blocks, ...) below.
    blocks.append(list(all_word_ids[start:start + CONTEXT_LEN]))
print(f"Number of training chunks: {len(blocks)}")

data = torch.tensor(blocks, dtype=torch.long)

# Shuffle the chunks (fixed seed, so this is reproducible) BEFORE the
# train/val split below. The corpus is source-blocked -- all of one
# source's words, then the next -- so without this, the validation slice
# (a prefix of `data`) would be drawn entirely from whichever source comes
# first (train_acts.json), not proportionally from all 4 sources. Training
# itself doesn't need this shuffle (the training loop below already
# reshuffles chunk order every epoch via torch.randperm), but the
# validation split is fixed once up front, so it needs its own shuffle.
val_split_seed = torch.Generator().manual_seed(42)
shuffled_chunk_order = torch.randperm(len(data), generator=val_split_seed)
data = data[shuffled_chunk_order]

# hold out a slice for validation / local testing -- a non-overlapping
# slice of (now-shuffled) chunks, so no word is ever shared between
# train_data and val_data
num_val = max(1, int(0.01 * len(data)))
val_data = data[:num_val]
train_data = data[num_val:]
print(f"Training chunks: {len(train_data)}, validation chunks: {len(val_data)}")

Number of training chunks: 115226
Training chunks: 114074, validation chunks: 1152


## 7. Building blocks for the research knobs

Before defining the model itself, here are the pieces `NORM_TYPE`,
`NORM_POSITION` and `OPTIMIZER_TYPE` (and `ATTENTION_TYPE`, for this
architecture) plug into. Every architecture in all 4 notebooks uses these
same building blocks, so results are comparable across notebooks.

In [11]:
class RMSNorm(nn.Module):
    """Like LayerNorm, but simpler: it only rescales by the root-mean-square
    of the values, it does not re-center the mean. Cheaper, and used in
    models like LLaMA."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


def make_norm(dim, norm_type):
    """norm_type is "layernorm", "rmsnorm" or "deepnorm".
    deepnorm reuses a plain LayerNorm -- what makes deepnorm special isn't
    the norm formula, it's the residual scaling (alpha) and weight init
    (beta) we apply around it below."""
    if norm_type == "rmsnorm":
        return RMSNorm(dim)
    else:  # "layernorm" or "deepnorm"
        return nn.LayerNorm(dim)


def deepnorm_alpha(n_layers):
    """Residual scaling factor from the DeepNet paper: alpha = (2N)^0.25."""
    return (2.0 * n_layers) ** 0.25


def deepnorm_init(output_projections, n_layers):
    """Scales the init of a sub-layer's OUTPUT projection(s) (e.g. attention
    out_proj, the feed-forward's second linear) by beta = (8N)^-0.25, so a
    deep post-norm stack stays stable to train. Only used when
    NORM_TYPE == "deepnorm"."""
    beta = (8.0 * n_layers) ** -0.25
    for module in output_projections:
        nn.init.xavier_normal_(module.weight, gain=beta)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

### Attention with `ATTENTION_TYPE` support (MHA / GQA / MQA)

`nn.MultiheadAttention` cannot give query heads fewer key/value heads than
query heads, so grouped-query and multi-query attention need a small
custom attention module. It still leans on PyTorch's built-in
`scaled_dot_product_attention` for the actual math (which automatically
uses a fast/FlashAttention-style kernel on capable GPUs — we don't need to
hand-write that part).

In [12]:
def n_kv_heads_for(attention_type, n_heads):
    """How many key/value heads to use for each attention type."""
    if attention_type == "mha":
        return n_heads              # every query head gets its own kv head
    if attention_type == "gqa":
        return max(1, n_heads // 4)  # groups of 4 query heads share one kv head
    if attention_type == "mqa":
        return 1                     # ALL query heads share one kv head
    raise ValueError(f"unknown attention_type {attention_type}")


class MultiHeadAttention(nn.Module):
    """Self-attention (x_kv=None) or cross-attention (x_kv given), with a
    configurable number of key/value heads so it can do plain multi-head
    attention (MHA), grouped-query attention (GQA) or multi-query
    attention (MQA)."""

    def __init__(self, d_model, n_heads, n_kv_heads, causal):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.causal = causal
        self.group_size = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def forward(self, x_q, x_kv=None, key_padding_mask=None):
        """key_padding_mask (optional): (batch, kv_len) bool, True = real
        (non-pad) key position. Used by batched generation, which
        left-pads shorter sequences so many prompts can be generated at
        once instead of one at a time -- see generate_continuations_batch
        below. Leave it None for training (no padding there)."""
        if x_kv is None:
            x_kv = x_q  # self-attention: queries and keys/values come from the same place

        batch_size, q_len, _ = x_q.shape
        kv_len = x_kv.shape[1]

        q = self.q_proj(x_q).view(batch_size, q_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x_kv).view(batch_size, kv_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x_kv).view(batch_size, kv_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if self.group_size > 1:
            # repeat each kv head so every query head in its group has a match
            k = k.repeat_interleave(self.group_size, dim=1)
            v = v.repeat_interleave(self.group_size, dim=1)

        if key_padding_mask is not None:
            positions = torch.arange(q_len, device=x_q.device)
            causal = positions.unsqueeze(1) >= positions.unsqueeze(0)  # (q_len, kv_len)
            # a key may be attended to if it's a real (non-pad) position, OR
            # it's the query's own position -- the "own position" exception
            # guarantees no row is fully masked out (which would make
            # softmax produce NaN). Pad-query rows are never actually read
            # (generation only looks at the last, always-real position), so
            # this is safe even though it "allows" a pad query to attend to
            # itself.
            key_ok = key_padding_mask.unsqueeze(1) | torch.eye(
                q_len, kv_len, dtype=torch.bool, device=x_q.device).unsqueeze(0)
            attn_mask = (causal.unsqueeze(0) & key_ok).unsqueeze(1)  # (batch, 1, q_len, kv_len)
            attn_out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=False)
        else:
            attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal)

        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, q_len, self.n_heads * self.head_dim)
        return self.out_proj(attn_out)

### Optimizer with `OPTIMIZER_TYPE` support (AdamW / Muon)

**Muon** (Keller Jordan) keeps a momentum buffer like normal SGD, but
before applying the update it "orthogonalizes" it (a fast Newton-Schulz
iteration pushes every singular value of the update matrix towards 1).
This only makes sense for 2D+ weight matrices — biases, embeddings and
norm weights still use plain AdamW underneath. This is the standard Muon
recipe (and why `build_optimizer` below actually returns two optimizers
wrapped so the training loop can use them like one).

In [13]:
def zeropower_via_newton_schulz(update_matrix, steps=5, eps=1e-7):
    """Orthogonalize a matrix (push its singular values towards 1) with a
    fast iterative approximation. This is the core trick behind Muon."""
    a, b, c = 3.4445, -4.7750, 2.0315
    x = update_matrix.float()
    flipped = x.size(-2) > x.size(-1)
    if flipped:
        x = x.transpose(-2, -1)
    x = x / (x.norm(dim=(-2, -1), keepdim=True) + eps)
    for _ in range(steps):
        a_term = x @ x.transpose(-2, -1)
        b_term = b * a_term + c * a_term @ a_term
        x = a * x + b_term @ x
    if flipped:
        x = x.transpose(-2, -1)
    return x


class Muon(torch.optim.Optimizer):
    """Simplified Muon optimizer. Only meant to be given 2D+ weight matrices."""

    def __init__(self, params, lr=0.02, momentum=0.95, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, ns_steps=ns_steps))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(p.grad)
                buf = state["momentum_buffer"]
                buf.mul_(group["momentum"]).add_(p.grad)
                update = zeropower_via_newton_schulz(buf, steps=group["ns_steps"])
                scale = max(1.0, p.grad.size(-2) / p.grad.size(-1)) ** 0.5
                p.add_(update.reshape(p.shape).to(p.dtype), alpha=-group["lr"] * scale)


class CombinedOptimizer:
    """Wraps two optimizers so the training loop can call .zero_grad() /
    .step() on it just like a single normal optimizer."""

    def __init__(self, optimizers):
        self.optimizers = optimizers

    def zero_grad(self):
        for opt in self.optimizers:
            opt.zero_grad()

    def step(self):
        for opt in self.optimizers:
            opt.step()

    def state_dict(self):
        return [opt.state_dict() for opt in self.optimizers]

    def load_state_dict(self, state_dicts):
        for opt, state in zip(self.optimizers, state_dicts):
            opt.load_state_dict(state)


def build_optimizer(model, optimizer_type, lr):
    """optimizer_type is "adamw" or "muon"."""
    if optimizer_type == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr)

    if optimizer_type == "muon":
        muon_params, adamw_params = [], []
        for name, p in model.named_parameters():
            # muon only works well on 2D+ "matrix" weights; everything else
            # (biases, norm weights, embedding tables) stays on adamw
            if p.ndim >= 2 and "emb" not in name.lower():
                muon_params.append(p)
            else:
                adamw_params.append(p)
        muon_opt = Muon(muon_params, lr=lr * 30)   # muon typically wants a bigger lr than adamw
        adamw_opt = torch.optim.AdamW(adamw_params, lr=lr)
        return CombinedOptimizer([muon_opt, adamw_opt])

    raise ValueError(f"unknown optimizer_type {optimizer_type}")

## 8. The model: Decoder-Only Transformer

A Transformer that uses self-attention, but is only allowed to look
at earlier words — never words that come after the current one. This
is how GPT-style models work: one stream of layers processes the
prompt and the continuation together, word by word.

In [14]:
class DecoderOnlyBlock(nn.Module):
    """One Transformer block: self-attention (causal) + feed-forward, wired
    up according to NORM_TYPE / NORM_POSITION / ATTENTION_TYPE."""

    def __init__(self, d_model, n_heads, d_ff, n_layers):
        super().__init__()
        # deepnorm is a specific POST-norm recipe -- it ignores NORM_POSITION
        self.norm_position = "post" if NORM_TYPE == "deepnorm" else NORM_POSITION

        n_kv_heads = n_kv_heads_for(ATTENTION_TYPE, n_heads)
        self.attn = MultiHeadAttention(d_model, n_heads, n_kv_heads, causal=True)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = make_norm(d_model, NORM_TYPE)
        self.norm2 = make_norm(d_model, NORM_TYPE)

        if NORM_TYPE == "deepnorm":
            self.alpha = deepnorm_alpha(n_layers)
            deepnorm_init([self.attn.out_proj, self.ff[2]], n_layers)
        else:
            self.alpha = 1.0

    def forward(self, x, key_padding_mask=None):
        if self.norm_position == "pre":
            x = x + self.attn(self.norm1(x), key_padding_mask=key_padding_mask)
            x = x + self.ff(self.norm2(x))
        else:  # "post" (also deepnorm, with the alpha residual scale)
            x = self.norm1(x * self.alpha + self.attn(x, key_padding_mask=key_padding_mask))
            x = self.norm2(x * self.alpha + self.ff(x))
        return x


class DecoderOnlyModel(nn.Module):
    """A Transformer that can only look backwards (causal / GPT-style)."""

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(context_len, d_model)
        self.blocks = nn.ModuleList([
            DecoderOnlyBlock(d_model, n_heads, d_ff, n_layers) for _ in range(n_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x, key_padding_mask=None):
        """key_padding_mask (optional): (batch, seq_len) bool, True = real
        (non-pad) position -- only used during batched generation, not
        training (training batches never have padding)."""
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        h = self.token_emb(x) + self.pos_emb(positions)
        for block in self.blocks:
            h = block(h, key_padding_mask=key_padding_mask)
        return self.fc_out(h)

## 9. Build the model

Sized so the parameter count lands in the 50M-300M range the brief
requires (printed below so you can check).

In [15]:
vocab_size = bpe_tokenizer.get_vocab_size()
model = DecoderOnlyModel(vocab_size, d_model=768, n_heads=12, n_layers=8,
                          d_ff=3072, context_len=CONTEXT_LEN)
model = model.to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {num_params:,} ({num_params / 1e6:.1f}M)")
if not (50_000_000 <= num_params <= 300_000_000):
    print("WARNING: this is outside the required 50M-300M range! "
          "Change d_model / n_layers above.")

Number of parameters: 118,674,496 (118.7M)


## 10. Resume from a checkpoint if one already exists

If this is the first time you're running this notebook (in this mode),
nothing to resume — training starts from scratch. If your Kaggle session
restarted earlier, this picks up right where you left off.

In [16]:
optimizer = build_optimizer(model, OPTIMIZER_TYPE, LEARNING_RATE)
start_epoch = 0

try_pull_checkpoint_from_hub(CHECKPOINT_PATH)  # in case a fresh session/account has no local copy

if os.path.exists(CHECKPOINT_PATH):
    print("Found an existing checkpoint, resuming from it...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    start_epoch = checkpoint["epoch"] + 1
    print(f"Resuming from epoch {start_epoch}")
else:
    print("No checkpoint found. Starting from scratch.")

No matching checkpoint on the Hub yet (RemoteEntryNotFoundError) -- starting fresh.
No checkpoint found. Starting from scratch.


## 11. Train the model

This is the main loop: show the model a batch of examples, see how wrong its
predictions were (the "loss"), and nudge its weights to do a little better
next time. We save a checkpoint after every epoch.

In [17]:
print(f"Training Decoder-Only Transformer for {NUM_EPOCHS} epoch(s), QUICK_TEST={QUICK_TEST}, "
      f"NORM_TYPE={NORM_TYPE}, NORM_POSITION={NORM_POSITION}, "
      f"OPTIMIZER_TYPE={OPTIMIZER_TYPE}...")

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    shuffled_order = torch.randperm(len(train_data))
    total_loss = 0.0
    num_batches = 0

    for start in range(0, len(train_data), BATCH_SIZE):
        batch_positions = shuffled_order[start:start + BATCH_SIZE]
        batch = train_data[batch_positions].to(device)

        inputs = batch[:, :-1]
        targets = batch[:, 1:]
        logits = model(inputs)

        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        if num_batches % 100 == 0:
            print(f"  epoch {epoch}, batch {num_batches}/{len(train_data)//BATCH_SIZE}: "
                  f"loss = {loss.item():.3f}")

    avg_loss = total_loss / max(1, num_batches)
    print(f"Epoch {epoch} finished. Average loss: {avg_loss:.3f}")

    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": epoch,
    }, CHECKPOINT_PATH)
    print("Checkpoint saved.")
    push_checkpoint_to_hub(CHECKPOINT_PATH)

print("Training complete!")

Training Decoder-Only Transformer for 2 epoch(s), QUICK_TEST=False, NORM_TYPE=layernorm, NORM_POSITION=pre, OPTIMIZER_TYPE=adamw...
  epoch 0, batch 100/14259: loss = 5.469
  epoch 0, batch 200/14259: loss = 5.658
  epoch 0, batch 300/14259: loss = 4.977
  epoch 0, batch 400/14259: loss = 5.250
  epoch 0, batch 500/14259: loss = 4.671
  epoch 0, batch 600/14259: loss = 5.342
  epoch 0, batch 700/14259: loss = 4.834
  epoch 0, batch 800/14259: loss = 4.721
  epoch 0, batch 900/14259: loss = 4.380
  epoch 0, batch 1000/14259: loss = 4.321
  epoch 0, batch 1100/14259: loss = 4.107
  epoch 0, batch 1200/14259: loss = 4.270
  epoch 0, batch 1300/14259: loss = 4.615
  epoch 0, batch 1400/14259: loss = 4.538
  epoch 0, batch 1500/14259: loss = 3.695
  epoch 0, batch 1600/14259: loss = 3.894
  epoch 0, batch 1700/14259: loss = 3.151
  epoch 0, batch 1800/14259: loss = 3.571
  epoch 0, batch 1900/14259: loss = 4.014
  epoch 0, batch 2000/14259: loss = 4.044
  epoch 0, batch 2100/14259: loss = 3

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed checkpoint to https://huggingface.co/kavincg23/cs781-checkpoints/blob/main/phase1_decoder_only_full_layernorm_pre_mha_adamw_checkpoint.pt
  epoch 1, batch 100/14259: loss = 2.430
  epoch 1, batch 200/14259: loss = 1.653
  epoch 1, batch 300/14259: loss = 2.539
  epoch 1, batch 400/14259: loss = 1.712
  epoch 1, batch 500/14259: loss = 2.126
  epoch 1, batch 600/14259: loss = 2.410
  epoch 1, batch 700/14259: loss = 2.811
  epoch 1, batch 800/14259: loss = 2.649
  epoch 1, batch 900/14259: loss = 2.806
  epoch 1, batch 1000/14259: loss = 2.032
  epoch 1, batch 1100/14259: loss = 2.211
  epoch 1, batch 1200/14259: loss = 2.742
  epoch 1, batch 1300/14259: loss = 2.380
  epoch 1, batch 1400/14259: loss = 2.406
  epoch 1, batch 1500/14259: loss = 1.958
  epoch 1, batch 1600/14259: loss = 2.090
  epoch 1, batch 1700/14259: loss = 2.596
  epoch 1, batch 1800/14259: loss = 2.136
  epoch 1, batch 1900/14259: loss = 2.212
  epoch 1, batch 2000/14259: loss = 2.018
  epoch 1, batch 2100/14

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed checkpoint to https://huggingface.co/kavincg23/cs781-checkpoints/blob/main/phase1_decoder_only_full_layernorm_pre_mha_adamw_checkpoint.pt
Training complete!


## 12. Check validation loss

This tells you how the model does on chunks it did NOT train on — a much
more honest signal than the training loss above.

In [18]:
model.eval()
total_val_loss = 0.0
num_val_batches = 0

with torch.no_grad():
    for start in range(0, len(val_data), BATCH_SIZE):
        batch = val_data[start:start + BATCH_SIZE].to(device)
        if len(batch) == 0:
            continue
        inputs = batch[:, :-1]
        targets = batch[:, 1:]
        logits = model(inputs)
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))
        total_val_loss += loss.item()
        num_val_batches += 1

avg_val_loss = total_val_loss / max(1, num_val_batches)
print(f"Validation loss: {avg_val_loss:.3f}")
print(f"Validation perplexity: {2.718281828 ** avg_val_loss:.1f}  (lower is better)")

Validation loss: 2.342
Validation perplexity: 10.4  (lower is better)


## 13. Generate the 128-word continuations for the test set

The competition wants exactly 128 words for every test prompt, containing
ONLY the generated continuation (not the original prompt).

In [19]:
def sample_next_word_id(logits, temperature):
    """Pick the next word. temperature=0 means always pick the single most
    likely word (greedy). A higher temperature makes the choice more random
    / creative. This is a required experiment in the brief - try changing it!"""
    if temperature == 0:
        return logits.argmax().item()
    probabilities = F.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probabilities, num_samples=1).item()

In [20]:
PAD_ID = 0   # the BPE tokenizer's special_tokens list put "<pad>" first, so its id is 0


def sample_next_word_ids_batch(logits_batch, temperature):
    """Same idea as picking one next word, but for a whole batch at once --
    logits_batch is (batch, vocab_size), returns (batch,) of next-token ids."""
    if temperature == 0:
        return logits_batch.argmax(dim=-1)
    probabilities = F.softmax(logits_batch / temperature, dim=-1)
    return torch.multinomial(probabilities, num_samples=1).squeeze(-1)


def generate_continuations_batch(prompt_texts, num_words=128, batch_size=16, verbose=True):
    """Generates continuations for MANY prompts at once instead of one at a
    time -- the model already supports batching, generating one prompt at a
    time (looping 1000+ times, each doing its own sequential forward
    passes) just wasn't using the GPU efficiently. Shorter prompts in a
    batch are LEFT-padded with PAD_ID up to CONTEXT_LEN, with a
    key_padding_mask telling attention to ignore the padding -- verified
    separately to produce IDENTICAL output to one-prompt-at-a-time
    generation, just much faster. Lower batch_size if you hit a CUDA
    out-of-memory error.

    With BPE, tokens != words, so like the single-prompt version, this
    keeps generating each sequence until it has STRICTLY MORE than
    num_words words, guaranteeing the num_words-th word is fully finished
    (not a subword fragment)."""
    model.eval()
    results = [None] * len(prompt_texts)
    max_new_tokens = num_words * 6   # safety cap so this can never loop forever

    for batch_start in range(0, len(prompt_texts), batch_size):
        batch_prompts = prompt_texts[batch_start:batch_start + batch_size]
        all_ids = [encode(p) for p in batch_prompts]
        start_lens = [len(ids) for ids in all_ids]
        done = [False] * len(batch_prompts)

        with torch.no_grad():
            for _ in range(max_new_tokens):
                if all(done):
                    break

                recent = [ids[-CONTEXT_LEN:] for ids in all_ids]
                pad_lens = [CONTEXT_LEN - len(r) for r in recent]
                padded = [[PAD_ID] * pad_lens[i] + recent[i] for i in range(len(recent))]
                mask = [[False] * pad_lens[i] + [True] * len(recent[i]) for i in range(len(recent))]

                x = torch.tensor(padded, dtype=torch.long).to(device)
                key_padding_mask = torch.tensor(mask, dtype=torch.bool).to(device)
                # fp16 autocast: T4 GPUs have dedicated fp16 tensor cores, so this
                # is close to a free speedup for inference (no gradients here, so
                # none of the usual mixed-precision-training complications apply).
                # A no-op on CPU (enabled=False).
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                    logits = model(x, key_padding_mask=key_padding_mask)
                next_ids = sample_next_word_ids_batch(logits[:, -1, :].float(), TEMPERATURE)

                for i in range(len(batch_prompts)):
                    if done[i]:
                        continue
                    all_ids[i].append(next_ids[i].item())
                    words_so_far = decode(all_ids[i][start_lens[i]:]).split()
                    if len(words_so_far) > num_words:
                        done[i] = True

        for i in range(len(batch_prompts)):
            words = decode(all_ids[i][start_lens[i]:]).split()
            words = words[:num_words]
            while len(words) < num_words:   # safety net (should basically never happen)
                words.append("<unk>")
            results[batch_start + i] = " ".join(words)

        if verbose:
            done_count = min(batch_start + batch_size, len(prompt_texts))
            print(f"  generated {done_count} / {len(prompt_texts)}")

    return results


def generate_continuation(prompt_text, num_words=128):
    """Single-prompt convenience wrapper around the batched version above
    (batch of 1) -- kept so any one-off call still works the same way."""
    return generate_continuations_batch([prompt_text], num_words=num_words,
                                         batch_size=1, verbose=False)[0]


# quick sanity check on one example before generating all of them
example_id = list(test_prompts.keys())[0]
example_continuation = generate_continuation(test_prompts[example_id])
print("Example continuation:")
print(example_continuation)
print()
print("Word count:", len(example_continuation.split()), "(should be 128)")

Example continuation:
not only to say that the Court has not only the discretion of the Court but also to decide the question of the question of the issue of the notification under challenge to the notification under challenge. The writ petition is not maintainable. The writ petition is dismissed. No order as to costs. Court No. - 1 Case :- CRIMINAL MISC. WRIT PETITION No. - 1188 of 2018 Petitioner :- Smt. And 3 Others Respondent :- State Of U.P.P. And 3 Others Counsel for Petitioner :- Vijay Bahadur Shivhare,Ambhiya Counsel for Respondent :- G.A. Hon'ble Ramesh Sinha,J. Hon'ble Dinesh Kumar Singh-I,J. Heard learned counsel for the petitioners and Sri N.K. Verma, learned A.G.G.A. for the State. This writ petition has been filed by the petitioners with a prayer

Word count: 128 (should be 128)


In [21]:
print("Generating continuations for all test prompts...")
test_ids = list(test_prompts.keys())
test_texts = [test_prompts[doc_id] for doc_id in test_ids]

GEN_BATCH_SIZE = 64   # lower this if you hit a CUDA out-of-memory error
generated_texts = generate_continuations_batch(test_texts, num_words=128,
                                                batch_size=GEN_BATCH_SIZE)
predictions = {doc_id: {"text": text} for doc_id, text in zip(test_ids, generated_texts)}

prediction_path = "/kaggle/working/prediction.json"
with open(prediction_path, "w") as f:
    json.dump(predictions, f, indent=2)

print(f"Saved {len(predictions)} predictions to {prediction_path}")

Generating continuations for all test prompts...
  generated 64 / 1000
  generated 128 / 1000
  generated 192 / 1000
  generated 256 / 1000
  generated 320 / 1000
  generated 384 / 1000
  generated 448 / 1000
  generated 512 / 1000
  generated 576 / 1000
  generated 640 / 1000
  generated 704 / 1000
  generated 768 / 1000
  generated 832 / 1000
  generated 896 / 1000
  generated 960 / 1000
  generated 1000 / 1000
Saved 1000 predictions to /kaggle/working/prediction.json


## 14. Check everything, then package `submission.zip`

Codabench needs `prediction.json` to sit at the ROOT of the zip file (not
inside a folder), and every official test ID must be present with no
extras. This cell checks both before zipping.

In [22]:
official_ids = set(test_prompts.keys())
predicted_ids = set(predictions.keys())

missing_ids = official_ids - predicted_ids
extra_ids = predicted_ids - official_ids

if missing_ids:
    print("PROBLEM: missing predictions for IDs:", missing_ids)
if extra_ids:
    print("PROBLEM: predictions include unknown IDs:", extra_ids)

wrong_word_count_ids = [doc_id for doc_id, pred in predictions.items()
                        if len(pred["text"].split()) != 128]
if wrong_word_count_ids:
    print("PROBLEM: these IDs do not have exactly 128 words:", wrong_word_count_ids)

if not missing_ids and not extra_ids and not wrong_word_count_ids:
    print("All checks passed: correct IDs, correct word counts.")

    submission_path = "/kaggle/working/submission.zip"
    with zipfile.ZipFile(submission_path, "w") as zip_file:
        # arcname="prediction.json" puts the file at the ROOT of the zip,
        # which is exactly what Codabench requires
        zip_file.write(prediction_path, arcname="prediction.json")

    print(f"Saved {submission_path}")
    print("If QUICK_TEST is still True, this is just a pipeline check -- "
          "don't submit it. Set QUICK_TEST = False, re-run the notebook, "
          "and package/submit THAT.")
    print("Download from the 'Output' section on the right side of this "
          "Kaggle notebook (you may need to click 'Save Version' first), "
          "then upload it to Codabench.")
else:
    print("Fix the problems above before packaging the submission.")

All checks passed: correct IDs, correct word counts.
Saved /kaggle/working/submission.zip
If QUICK_TEST is still True, this is just a pipeline check -- don't submit it. Set QUICK_TEST = False, re-run the notebook, and package/submit THAT.
Download from the 'Output' section on the right side of this Kaggle notebook (you may need to click 'Save Version' first), then upload it to Codabench.


## 15. (Optional) Local evaluation with BERTScore / ROUGE-L / BLEU-4

The real test answers are private, so we build our own small
prompt/answer pairs from the validation chunks we held out in Step 6, and
score our model against those. This won't match the real leaderboard
number exactly, but it lets you compare this run (this architecture AND
these NORM_TYPE / NORM_POSITION / ATTENTION_TYPE / OPTIMIZER_TYPE
settings) against your other runs before spending a Codabench submission.

In [23]:
!pip install -q bert-score rouge-score sacrebleu

from bert_score import score as bert_score
from rouge_score import rouge_scorer
import sacrebleu

print("Building a local prompt/answer set from validation data...")
# Decode the WHOLE held-out validation stream into words first, then slice
# by word count -- NOT chunk-by-chunk. With BPE, one CONTEXT_LEN-token
# chunk decodes to FEWER than CONTEXT_LEN words (tokens != words), so the
# old per-chunk "len(words) >= 640" check always failed and silently built
# zero examples. Concatenating first works correctly for BPE and word-level
# tokenizers alike.
val_ids = torch.cat([chunk for chunk in val_data]).tolist()
val_words = decode(val_ids).split()

local_prompts = []
local_references = []
example_len = 512 + 128
for start in range(0, len(val_words) - example_len, example_len):
    words = val_words[start:start + example_len]
    local_prompts.append(" ".join(words[:512]))
    local_references.append(" ".join(words[512:example_len]))

print(f"Built {len(local_prompts)} local evaluation examples")
if len(local_prompts) == 0:
    print("Not enough validation data to build even one local eval example "
          "(common in QUICK_TEST mode, which uses very little data) -- "
          "run Step 16 below only once this prints > 0, e.g. after a real "
          "QUICK_TEST=False run.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.1 MB/s eta 0:00:00
Building a local prompt/answer set from validation data...
Built 938 local evaluation examples


In [24]:
if len(local_prompts) == 0:
    print("Skipping local evaluation -- no local eval examples were built above.")
else:
    local_predictions = generate_continuations_batch(local_prompts, num_words=128,
                                                       batch_size=GEN_BATCH_SIZE, verbose=False)

    precision, recall, f1 = bert_score(local_predictions, local_references, lang="en", verbose=False)
    print("BERTScore F1 (primary metric):", f1.mean().item())

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_scores = [rouge.score(ref, pred)["rougeL"].fmeasure
                    for pred, ref in zip(local_predictions, local_references)]
    print("ROUGE-L F1:", sum(rouge_scores) / len(rouge_scores))

    bleu = sacrebleu.corpus_bleu(local_predictions, [local_references])
    print("BLEU-4:", bleu.score)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore F1 (primary metric): 0.7862679362297058
ROUGE-L F1: 0.17412000761823965
BLEU-4: 5.009825217098057


## What to try next (this is where the real grade comes from)

The brief cares more about what you tried and learned than the raw score.
Once this notebook's real run (`QUICK_TEST = False`) works end to end, easy
things to change and re-run:

- **`NORM_TYPE` / `NORM_POSITION`** — try `rmsnorm` and `deepnorm`, and
  `pre` vs `post`. Compare training stability (does the loss curve look
  clean or does it spike?) and the local eval scores.
- **`ATTENTION_TYPE`** (this notebook, if it has one) — try `gqa` and
  `mqa` and see how much of the score they trade away for fewer
  key/value heads.
- **`OPTIMIZER_TYPE`** — compare `muon` against `adamw`: does it converge
  faster (look at the loss printed every 100 batches)?
- **`TEMPERATURE`** in Settings — try 0.7 or 1.0 instead of 0 (greedy) and
  compare the generated text and the local eval scores.
- **`VOCAB_SIZE`** — a bigger vocabulary means fewer `<unk>` words but a
  bigger embedding table.
- **`MAX_WORDS_FOR_TRAINING`** in the `else` branch of Settings — push this
  up as your GPU/time budget allows.
- **Which of the 4 sources you train on** — right now `TRAIN_FILES` walks
  through all 4 in a fixed order. Try filtering to just Supreme Court
  judgments, or just Acts, and see what changes.
- **Model size / `NUM_EPOCHS` / `LEARNING_RATE` / `BATCH_SIZE`** — the classic knobs.
- Once all 4 of your architecture notebooks have a real (`QUICK_TEST = False`)
  run with a local BERTScore/ROUGE-L/BLEU-4 number (Step 15), compare them
  and decide which model(s) you actually submit to Codabench and carry into
  Phase 2.

Remember: **minimum 4 real submissions** to Codabench before the Phase 1
deadline, and keep notes on what you tried and what happened (including
these new norm/attention/optimizer settings) in each notebook — that's
what goes into your final report.